# Explicativo del preprocesamiento y modelado con cross validation

- Correr los scripts `03_preprocessing.py`  y  `04b_pre_cv_split.py`
- Ejecutar la notebook `05b_modeling_cv.ipynb`

In [ ]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Asumimos que 'df_train' es tu 85% de datos (Train + Validation temporal combinados)
# y que ya pasó por el bloque de limpieza y feature engineering.
# df_train = ... 

X = df_train.drop(columns=['is_canceled', 'booking_date', 'arrival_date'])
y = df_train['is_canceled']

# 2. Definición del Preprocesador
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]
vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num'
]
vars_binarias = ['is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], remainder='drop'
)

# 3. Definición de los Pipelines (Modelos)
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# Usamos n_estimators=50 (o 100) y n_jobs=-1 para usar todos los núcleos del procesador
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced'))
])

# 4. Configuración de la Validación Cruzada Temporal (5 iteraciones)
tscv = TimeSeriesSplit(n_splits=5)

# 5. Ejecución y cálculo de métricas (ROC-AUC)
print("Evaluando Regresión Logística...")
scores_lr = cross_val_score(pipeline_lr, X, y, cv=tscv, scoring='roc_auc', n_jobs=-1)

print("Evaluando Random Forest...")
scores_rf = cross_val_score(pipeline_rf, X, y, cv=tscv, scoring='roc_auc', n_jobs=-1)

# 6. Resultados
print(f"\n--- Resultados de Validación Cruzada (ROC-AUC) ---")
print(f"Regresión Logística: {scores_lr.mean():.4f} (+/- {scores_lr.std() * 2:.4f})")
print(f"Random Forest:       {scores_rf.mean():.4f} (+/- {scores_rf.std() * 2:.4f})")

### Implementación de Validación Cruzada (Cross-Validation)
Dado que nuestro problema tiene una fuerte dependencia temporal (ordenado por fecha de reserva), utilizar la validación cruzada estándar (KFold o StratifiedKFold) incurriría en Data Leakage, ya que entrenaría modelos con datos del futuro para predecir el pasado.

Para implementar Cross-Validation correctamente en este contexto, se debe utilizar TimeSeriesSplit de scikit-learn. Este método crea particiones de tamaño creciente, asegurando que los datos de entrenamiento siempre precedan cronológicamente a los datos de validación en cada iteración.

El TimeSeriesSplit funciona mediante una técnica llamada ventana expansiva (expanding window). En lugar de barajar los datos o tomar bloques aislados y aleatorios como hace la validación cruzada tradicional, ancla el inicio del entrenamiento en el primer día de tu dataset y va "creciendo" cronológicamente hacia el futuro paso a paso.

Imagina que dividimos toda tu línea de tiempo ordenada por booking_date (ese 85% inicial que separamos) en 6 bloques secuenciales de tamaño similar. Si pedimos 5 particiones (n_splits=5), el algoritmo avanza en el tiempo de esta manera:

Iteración 1: Entrena con el Bloque 1. Valida prediciendo el Bloque 2.

Iteración 2: Entrena con los Bloques 1 y 2. Valida prediciendo el Bloque 3.

Iteración 3: Entrena con los Bloques 1, 2 y 3. Valida prediciendo el Bloque 4.

Iteración 4: Entrena con los Bloques 1, 2, 3 y 4. Valida prediciendo el Bloque 5.

Iteración 5: Entrena con los Bloques 1, 2, 3, 4 y 5. Valida prediciendo el Bloque 6.

Para hacerlo visualmente explícito, la estructura de particiones se ve exactamente así:

| Partición	| Datos de Entrenamiento (Pasado) |	Datos de Validación (Futuro a predecir)	| Datos futuros ignorados temporalmente |
|---|---|---|---|
| Fold 1	| [ Bloque 1 ]	| [ Bloque 2 ]	|[ 3 ] [ 4 ] [ 5 ] [ 6 ] |
| Fold 2	| [ Bloque 1 ] [ Bloque 2 ] |	[ Bloque 3 ]	| [ 4 ] [ 5 ] [ 6 ] |
| Fold 3	| [ Bloque 1 ] [ Bloque 2 ] [ Bloque 3 ]	 | [ Bloque 4 ]	| [ 5 ] [ 6 ] |
| Fold 4	| [ Bloque 1 ] [ Bloque 2 ] [ Bloque 3 ] [ Bloque 4 ]	| [ Bloque 5 ]	| [ 6 ] |
| Fold 5	| [ Bloque 1 ] | [ Bloque 2 ] [ Bloque 3 ] [ Bloque 4 ] [ Bloque 5 ]	[ Bloque 6 ]	| (Ninguno) |



Cuando hicimos el primer corte simple (Train/Validation/Test), básicamente ejecutamos de forma manual un proceso similar a un único fold: tomamos el pasado lejano para entrenar y evaluamos en el futuro inmediato.

El valor inmenso de hacer esto 5 veces a lo largo de la historia de reservas es que evaluamos la estabilidad matemática del algoritmo. Si el Random Forest rinde excelente en el Fold 1 pero colapsa en el Fold 4, significa que ocurrió un evento anómalo en el tiempo (como un cambio en las políticas de cancelación del hotel) que el modelo no sabe manejar. Si el rendimiento (ROC-AUC) se mantiene estable a lo largo de las 5 ventanas, tienes evidencia irrefutable de que el modelo generaliza bien las reglas de negocio.